In [48]:
prompt = """
Analyze this mirror selfie for a digital wardrobe.

Identify every clearly visible clothing and footwear item being worn by the person.

For every item, return:

- category
- type: top, bottom, outerwear, footwear, accessory
- primary_color
- secondary_color
- pattern
- confidence
- bounding_box

The bounding_box must be provided as:

[ymin, xmin, ymax, xmax]

Coordinates must be normalized from 0 to 1000.

The bounding box should tightly surround the clothing item itself.
Do not include unnecessary background, the mirror, the phone, or unrelated body parts where possible.

Only identify clothing and footwear that are actually being worn.

Do not identify:
- the person
- skin
- hair
- mirror
- phone
- furniture
- walls
- background objects

If an item is partially obscured but clearly identifiable, include it and reduce the confidence score accordingly.

Do not invent items that cannot be clearly seen.

Return ONLY valid JSON in exactly this format (no need to wrap in a JSON markdown code bloc, just the plain text):

{
  "items": [
    {
      "category": "hoodie",
      "type": "top",
      "primary_color": "black",
      "secondary_color": null,
      "pattern": "plain",
      "confidence": 0.95,
      "bounding_box": [200, 300, 550, 700]
    }
  ]
}
"""

In [49]:
import os
import json
from google import genai
from google.genai import types
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

def analyze_outfit(image_path: str):
    try:
        with open(image_path, "rb") as image_file:
            response = client.models.generate_content(
                model="gemini-3.5-flash",
                contents=[
                    types.Part.from_text(text=prompt),
                    types.Part.from_bytes(
                        data=image_file.read(),
                        mime_type="image/jpeg"
                    )
                ],
            )

        return json.loads(response.text)
    except Exception as error:
        return {"error": str(error)}

In [50]:
result = analyze_outfit("/Users/guestuser/Documents/Projects/what-do-i-wear-today/experiments/test-images/test_1.png")

In [51]:
print(json.dumps(result, indent=2))

{
  "items": [
    {
      "category": "shirt",
      "type": "top",
      "primary_color": "blue",
      "secondary_color": null,
      "pattern": "plain",
      "confidence": 0.98,
      "bounding_box": [
        341,
        0,
        779,
        649
      ]
    },
    {
      "category": "pants",
      "type": "bottom",
      "primary_color": "black",
      "secondary_color": null,
      "pattern": "plain",
      "confidence": 0.95,
      "bounding_box": [
        730,
        125,
        1000,
        565
      ]
    },
    {
      "category": "glasses",
      "type": "accessory",
      "primary_color": "grey",
      "secondary_color": null,
      "pattern": "plain",
      "confidence": 0.9,
      "bounding_box": [
        279,
        183,
        332,
        359
      ]
    }
  ]
}
